In [22]:
!hdfs dfs -mkdir -p /datasets/accidentes/resultados/cruce_aire
!hdfs dfs -ls /datasets/accidentes/csv
!hdfs dfs -ls /dataset/calidad_aire_madrid

Found 1 items
-rw-r--r--   3 root supergroup   19043901 2026-10-23 09:35 /datasets/accidentes/csv/TABLA_ACCIDENTES_24.csv
Found 2 items
-rw-r--r--   3 root supergroup   20166384 2026-10-01 12:24 /dataset/calidad_aire_madrid/calidad_aire_madrid.csv
-rw-r--r--   3 root supergroup    3828371 2026-10-01 13:19 /dataset/calidad_aire_madrid/calidad_aire_madrid_20000.csv


In [23]:
%%writefile mapper_cruce.py
#!/usr/bin/env python3

import sys
import os
import csv

# Identifica el fichero que Hadoop está procesando
ruta = (
    os.environ.get("mapreduce_map_input_file")
    or os.environ.get("map_input_file")
    or ""
).lower()

es_accidentes = "accidentes" in ruta
es_calidad_aire = "calidad_aire" in ruta

if not es_accidentes and not es_calidad_aire:
    sys.stderr.write(
        f"No se reconoce el fichero de entrada: {ruta}\n"
    )
    sys.exit(1)

for linea in sys.stdin:
    try:
        fila = next(csv.reader([linea], delimiter=";"))

        # Ignorar líneas vacías
        if not fila:
            continue

        # -----------------------------------------
        # Fichero de accidentes de tráfico
        # -----------------------------------------
        if es_accidentes:
            if fila[0].strip().upper() == "ID_ACCIDENTE":
                continue

            # Índices del CSV de accidentes
            provincia = fila[5].strip().zfill(2)
            mes = int(fila[2].strip())

            # Solo accidentes de la provincia de Madrid
            if provincia != "28":
                continue

            if 1 <= mes <= 12:
                # Clave: mes
                # Valor: accidente
                print(f"{mes:02d}\tACC|1")

        # -----------------------------------------
        # Fichero de calidad del aire
        # -----------------------------------------
        elif es_calidad_aire:
            if fila[0].strip().upper() == "PROVINCIA":
                continue

            # Índices de las columnas del CSV de calidad del aire
            magnitud = fila[3].strip()
            mes = int(fila[6].strip())

            if not (1 <= mes <= 12):
                continue

            # Las columnas H01-H24 están en posiciones pares
            # desde el índice 8: 8, 10, 12, ..., 54.
            valores = []

            for indice in range(8, min(56, len(fila)), 2):
                dato = fila[indice].strip()

                if dato == "":
                    continue

                try:
                    valor = float(dato)

                    # Excluir valores no finitos y marcadores negativos
                    if valor >= 0 and valor < float("inf"):
                        valores.append(valor)

                except ValueError:
                    continue

            if valores:
                suma = sum(valores)
                cantidad = len(valores)

                # Clave: mes
                # Valor: magnitud, suma de mediciones y cantidad
                print(
                    f"{mes:02d}\tAIR|{magnitud}|"
                    f"{suma}|{cantidad}"
                )

    except (ValueError, IndexError, csv.Error) as error:
        sys.stderr.write(
            f"Registro ignorado: {error}\n"
        )


Writing mapper_cruce.py


In [31]:
%%writefile reducer_cruce.py

#!/usr/bin/env python3

import sys

# Descripciones de las magnitudes.
# Completar o ajustar según los códigos presentes en el CSV.
DESCRIPCIONES = {
    "1": "Dióxido de azufre (SO2)",
    "6": "Monóxido de carbono (CO)",
    "7": "Monóxido de nitrógeno (NO)",
    "8": "Dióxido de nitrógeno (NO2)",
    "9": "Partículas en suspensión PM2.5",
    "10": "Partículas en suspensión PM10",
    "12": "Óxidos de nitrógeno (NOx)",
    "14": "Ozono (O3)",
}

mes_actual = None
accidentes = 0
contaminantes = {}
cabecera_emitida = False


def mostrar_resultados(mes, accidentes, contaminantes):
    global cabecera_emitida

    if mes is None:
        return

    # Mostrar la cabecera una sola vez
    if not cabecera_emitida:
        print(
            "Mes\tAccidentes en Madrid\tCódigo de magnitud\t"
            "Descripción de la magnitud\tMedia mensual\t"
            "N.º de mediciones"
        )
        cabecera_emitida = True

    # Mostrar una fila por cada contaminante
    for magnitud in sorted(
        contaminantes,
        key=lambda x: int(x) if x.isdigit() else x
    ):
        suma, cantidad = contaminantes[magnitud]

        if cantidad > 0:
            media = suma / cantidad
            descripcion = DESCRIPCIONES.get(
                magnitud,
                "Magnitud sin descripción en el diccionario"
            )

            print(
                f"{mes}\t{accidentes}\t{magnitud}\t"
                f"{descripcion}\t{media:.4f}\t{cantidad}"
            )


for linea in sys.stdin:
    linea = linea.rstrip("\n")

    try:
        clave, valor = linea.split("\t", 1)
    except ValueError:
        continue

    if mes_actual is not None and clave != mes_actual:
        mostrar_resultados(
            mes_actual, accidentes, contaminantes
        )
        accidentes = 0
        contaminantes = {}

    mes_actual = clave
    partes = valor.split("|")

    if partes[0] == "ACC":
        accidentes += int(partes[1])

    elif partes[0] == "AIR":
        magnitud = partes[1]
        suma = float(partes[2])
        cantidad = int(partes[3])

        if magnitud not in contaminantes:
            contaminantes[magnitud] = [0.0, 0]

        contaminantes[magnitud][0] += suma
        contaminantes[magnitud][1] += cantidad


mostrar_resultados(mes_actual, accidentes, contaminantes)

Overwriting reducer_cruce.py


In [28]:
!chmod +x mapper_cruce.py.py reducer_cruce.py


chmod: cannot access 'mapper_cruce.py.py': No such file or directory


In [32]:
!hdfs dfs -rm -r -f /datasets/accidentes/resultados/cruce_aire

!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
  -files mapper_cruce.py,reducer_cruce.py \
  -mapper "python3 mapper_cruce.py" \
  -reducer "python3 reducer_cruce.py" \
  -input /datasets/accidentes/csv/TABLA_ACCIDENTES_24.csv \
  -input /dataset/calidad_aire_madrid/calidad_aire_madrid.csv \
  -output /datasets/accidentes/resultados/cruce_aire

Deleted /datasets/accidentes/resultados/cruce_aire
packageJobJar: [/tmp/hadoop-unjar2740592523224252379/] [] /tmp/streamjob8389030150892699511.jar tmpDir=null
2026-10-26 10:54:43,447 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-10-26 10:54:43,804 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-10-26 10:54:44,407 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1792745247675_0004
2026-10-26 10:54:46,053 INFO mapred.FileInputFormat: Total input files to process : 2
2026-10-26 10:54:46,239 INFO mapreduce.JobSubmitter: number of splits:2
2026-10-26 10:54:46,525 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1792745247675_0004
2026-10-26 10:54:46,525 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-26 10:54:46,783 INFO conf.Configuration: resource-types.xml not found


In [33]:
!hdfs dfs -cat /datasets/accidentes/resultados/cruce_aire/part-*

Mes	Accidentes en Madrid	Código de magnitud	Descripción de la magnitud	Media mensual	N.º de mediciones
01	1145	1	Dióxido de azufre (SO2)	3.4677	4464
01	1145	6	Monóxido de carbono (CO)	0.3469	5952
01	1145	7	Monóxido de nitrógeno (NO)	10.8099	34200
01	1145	8	Dióxido de nitrógeno (NO2)	27.7408	34200
01	1145	9	Partículas en suspensión PM2.5	7.3509	16320
01	1145	10	Partículas en suspensión PM10	13.1931	20808
01	1145	12	Óxidos de nitrógeno (NOx)	44.3227	34200
01	1145	14	Ozono (O3)	39.5934	19320
01	1145	20	Magnitud sin descripción en el diccionario	2.0211	5952
01	1145	30	Magnitud sin descripción en el diccionario	0.5365	5952
01	1145	35	Magnitud sin descripción en el diccionario	0.3322	5952
02	1100	1	Dióxido de azufre (SO2)	2.9144	4032
02	1100	6	Monóxido de carbono (CO)	0.3415	5376
02	1100	7	Monóxido de nitrógeno (NO)	10.2279	30912
02	1100	8	Dióxido de nitrógeno (NO2)	28.6716	30912
02	1100	9	Partículas en suspensión PM2.5	8.1039	14760
02	1100	10	Partículas en suspensión PM10	14.9443	18792
02	1